<a href="https://colab.research.google.com/github/Rutheniumruto/Signal-Processing-Lab-/blob/main/CN240_SF251_Lab04_Extracting_Signal_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratory 4 — Extracting Useful Signal Features

**CN240 Data Science for Signal Processing · SF251 Introduction to Data Science**
Week 4 · Semester 1, Academic Year 2026 · Dr Surasak Phetmanee

---

| | |
|---|---|
| **Name** | *Ruth Benedicta Susanto* |
| **Student ID** | *6710755239* |
| **Section** | *7500001* |

## What you will do today

Same signals, new questions. Week 3 made the ECG *clean* — today you make it *speak*:
mean, σ and RMS to describe it, peak detection to time-stamp every heartbeat, and correlation
to compare signals. The end product is the course's first real **feature matrix** — one row per
signal, one column per feature — exactly the kind of table week 8's classifiers will train on.

**Deliverables (due before week 5, pushed to your GitHub repository):**
1. **The feature table** — one row per signal: mean · σ · RMS · n_peaks · rate/min
2. **The honesty plot** — raw vs cleaned ECG side by side, detected peaks marked on both
3. **One insight** — 2–3 sentences: something the *numbers* told you that the waveform plot alone did not

> **House rule (from today's lecture, forever):** never report a correlation you have not scatter-plotted.


## Setup — run these cells first, do not edit

Cells 2–3 recreate the week-3 ECG *exactly* (same seed, same damage) and apply the Lab 3 recipe in
condensed form, so `ecg_clean` here matches the signal you produced last week.
*(Prefer your own Lab 3 pipeline? Paste it over cell 3 — everything downstream only needs `ecg_clean`.)*


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import medfilt, find_peaks

plt.rcParams['figure.figsize'] = (12, 3.5)
plt.rcParams['axes.grid'] = True
print("NumPy", np.__version__, "· Pandas", pd.__version__)

In [ ]:
# --- ECG setup: identical to Lab 3 (run once, do not edit) -------------------
FS = 360                                        # sampling rate, Hz
rng = np.random.default_rng(240)

try:                                            # real 5-minute MIT-BIH excerpt
    from scipy.datasets import electrocardiogram
    clean_reference = electrocardiogram()[:60 * FS]          # first 60 s
    SOURCE = "MIT-BIH (real recording)"
except Exception:                               # offline fallback: synthetic ECG
    t = np.arange(0, 60, 1 / FS)
    beat = 1.1 * np.exp(-((t % 0.85) - 0.28) ** 2 / 0.0009)
    clean_reference = (beat - 0.25 * np.exp(-((t % 0.85) - 0.38) ** 2 / 0.004)
                       + 0.03 * rng.standard_normal(t.size))
    SOURCE = "synthetic (offline fallback)"

n = clean_reference.size
t = np.arange(n) / FS
dirty = clean_reference.copy()
dirty += 0.35 * np.sin(2 * np.pi * 0.25 * t)                 # baseline wander
spike_idx = rng.choice(n, size=40, replace=False)            # electrode pops
dirty[spike_idx] += rng.choice([-1, 1], 40) * rng.uniform(3, 5, 40)
for start in rng.choice(n - 80, size=25, replace=False):     # dropouts
    dirty[start : start + rng.integers(5, 60)] = np.nan
dirty = pd.Series(dirty)
print(f"ecg loaded from {SOURCE}: {n} samples = {n/FS:.0f} s at {FS} Hz")

In [ ]:
# --- Lab 3's recipe, condensed (given — or paste your own pipeline) ----------
x = dirty.interpolate(limit_direction="both")            # repair holes
x = pd.Series(medfilt(x, 5))                             # kill spikes
x = x.rolling(9, center=True, min_periods=1).mean()      # smooth
baseline = x.rolling(int(1.0 * FS), center=True, min_periods=1).median()
x = x - baseline                                         # remove wander
ecg_clean = ((x - x.mean()) / x.std()).values            # normalise (z-units!)

plt.plot(t[:5 * FS], ecg_clean[:5 * FS], linewidth=0.8)
plt.title("ecg_clean — first 5 s (z-units)"); plt.xlabel("time [s]"); plt.show()
print("NOTE: ecg_clean is z-scored — amplitudes are in standard deviations, not mV")

In [ ]:
# --- Walking accelerometer, 60 s at 50 Hz (run once, do not edit) ------------
# A fresh walking recording in the same format as Lab 2's accel.csv.
# The subject walks at 1.8 steps/s; the arms swing at half that rate.
FSA = 50
rng_a = np.random.default_rng(240)
ta = np.arange(0, 60, 1 / FSA)
swing = np.sin(2 * np.pi * 0.9 * ta)                         # arm swing (stride rate)
walk = pd.DataFrame({
    "time_s": ta,
    "ax_g": 0.50 * swing + 0.20 * rng_a.standard_normal(ta.size),
    "ay_g": 0.35 * swing + 0.20 * rng_a.standard_normal(ta.size),
    "az_g": 1.0 + 0.30 * np.abs(np.sin(np.pi * 1.8 * ta)) ** 8   # step impacts
             + 0.05 * rng_a.standard_normal(ta.size),
})
walk.plot(x="time_s", y=["ax_g", "ay_g", "az_g"], subplots=True, figsize=(12, 5),
          xlim=(0, 10), title=["ax — arm swing", "ay — arm swing", "az — gravity + steps"])
plt.tight_layout(); plt.show()

---
## Mission 1 — Describe: one honest number per question

Write `describe()` once, use it on everything. Remember the lecture's reading order:
**mean** = where the signal sits · **σ** = how hard it moves · **RMS** = how big it effectively is.


In [ ]:
def describe(x) -> dict | None:
    """Return {'mean': ..., 'std': ..., 'rms': ...} for a 1-D signal."""
    # TODO: np.mean, np.std, and rms = sqrt(mean(x**2))  (slide 6)
    return None


d = describe(walk["az_g"].values)
if d is None:
    print("describe() not filled in yet")
else:
    for name, sig in [("ecg_clean", ecg_clean), ("ax_g", walk["ax_g"].values),
                      ("ay_g", walk["ay_g"].values), ("az_g", walk["az_g"].values)]:
        s = describe(sig)
        print(f"{name:>10}: mean {s['mean']:>7.3f} | std {s['std']:>6.3f} | rms {s['rms']:>6.3f}")

In [ ]:
# Checkpoint — the lecture's identity: RMS² = mean² + σ²  (slide 6)
d = describe(walk["az_g"].values)
if d is None:
    print("finish describe() first")
else:
    lhs = d["rms"] ** 2
    rhs = d["mean"] ** 2 + d["std"] ** 2
    print(f"RMS² = {lhs:.5f}   mean² + σ² = {rhs:.5f}   equal: {np.isclose(lhs, rhs)}")

**Q1 — read your own table:** `ecg_clean` has mean ≈ 0 while `az_g` has mean ≈ 1.08.
Neither is an accident — explain where each number comes from. *(Hint: one was engineered in week 3;
the other is a law of physics.)*

*(type your answer here)*


---
## Mission 2 — Detect: every heartbeat, time-stamped

### 2a — the naive attempt *(given — run and be suspicious)*

Slide 8's rules, taken literally: a height threshold at half the maximum, plus the distance rule.


In [ ]:
p_naive, _ = find_peaks(ecg_clean, height=0.5 * ecg_clean.max(), distance=int(0.25 * FS))
print(f"height rule finds {len(p_naive)} beats in 60 s  ->  HR ~ {len(p_naive):d} bpm")
print("Plausible-looking number. But is it TRUE? Keep it — 2b checks it.")

In [ ]:
# TODO (2b): the robust version — prominence instead of height  (slide 9's third parameter)
#   peaks, _ = find_peaks(ecg_clean, prominence=1.5, distance=int(0.25 * FS))
#   then: rr = np.diff(peaks) / FS          (seconds between beats)
#         hr = 60 / rr.mean()               (beats per minute)
#         sdnn_ms = rr.std() * 1000         (HRV: SD of the intervals, in ms)

peaks = None
hr = None
sdnn_ms = None

if peaks is None:
    print("2b not filled in yet")
else:
    print(f"prominence rule: {len(peaks)} beats | HR = {hr:.1f} bpm | SDNN = {sdnn_ms:.0f} ms")

In [ ]:
# Checkpoint (given): the pristine reference is the judge ---------------------
if peaks is None:
    print("finish 2b first")
else:
    ref = (clean_reference - clean_reference.mean()) / clean_reference.std()
    p_ref, _ = find_peaks(ref, prominence=1.5, distance=int(0.25 * FS))
    print(f"undamaged reference: {len(p_ref)} beats  |  your cleaned signal: {len(peaks)} beats")
    print("If those two agree (and 2a's count is far below), you have just watched the")
    print("height rule miss real beats: this MIT-BIH patient's beat SHAPES vary, so many")
    print("true R-peaks sit below half-of-max. Prominence measures local stand-out instead.")

In [ ]:
# Deliverable 2 (given): the honesty plot — raw vs cleaned, same detector ------
if peaks is None:
    print("finish 2b first")
else:
    raw_z = np.nan_to_num(dirty.values)
    raw_z = (raw_z - raw_z.mean()) / raw_z.std()
    p_raw, _ = find_peaks(raw_z, height=0.5 * raw_z.max(), distance=int(0.25 * FS))

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    sl = slice(0, 10 * FS)
    axes[0].plot(t[sl], raw_z[sl], linewidth=0.7, color="gray")
    axes[0].plot(t[p_raw[p_raw < 10 * FS]], raw_z[p_raw[p_raw < 10 * FS]], "rx", markersize=8)
    axes[0].set_title(f"RAW + height rule: {len(p_raw)} 'beats' in 60 s — the spikes own the scale")
    axes[1].plot(t[sl], ecg_clean[sl], linewidth=0.7, color="darkred")
    axes[1].plot(t[peaks[peaks < 10 * FS]], ecg_clean[peaks[peaks < 10 * FS]], "go", markersize=6)
    axes[1].set_title(f"CLEANED + prominence rule: {len(peaks)} beats — HR {hr:.0f} bpm")
    axes[1].set_xlabel("time [s]")
    plt.tight_layout(); plt.show()

**Q2 — three detectors, three numbers:** 2a (height on clean), 2b (prominence on clean) and the
raw run in the plot all disagree. Which number would you hand to a doctor, and what is your *evidence* —
not your feeling — from this notebook? Also: the SDNN you measured is genuinely large. This record
really is arrhythmic — is a big HRV here a bug in your pipeline or a fact about the patient? How do you know?

*(type your answer here)*


---
## Mission 3 — Correlate: do the arms swing together?

Three axis pairs, three r values — and the house rule: **every r gets its scatter plot.**


In [ ]:
# TODO (3.1): Pearson's r for the three pairs  (slide 12: np.corrcoef(a, b)[0, 1])
r_xy = None
r_xz = None
r_yz = None

if r_xy is None:
    print("3.1 not filled in yet")
else:
    print(f"r(ax, ay) = {r_xy:+.3f}   <- arm swing appears in BOTH horizontal axes")
    print(f"r(ax, az) = {r_xz:+.3f}")
    print(f"r(ay, az) = {r_yz:+.3f}")

In [ ]:
# The mandatory scatter plot (given — because the house rule is not optional)
if r_xy is None:
    print("finish 3.1 first")
else:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax_, (a, b, r) in zip(axes, [("ax_g", "ay_g", r_xy), ("ax_g", "az_g", r_xz),
                                     ("ay_g", "az_g", r_yz)]):
        ax_.plot(walk[a], walk[b], ".", markersize=2, alpha=0.5)
        ax_.set_xlabel(a); ax_.set_ylabel(b); ax_.set_title(f"r = {r:+.3f}")
    plt.tight_layout(); plt.show()
    print("One tilted cloud, two round blobs — the picture and the numbers must tell the same story.")

---
## Mission 4 — Deliverable 1: the feature table

One row per signal, one column per feature. For peaks, each signal needs its own physiology:
hearts cannot beat twice in 0.25 s — but a walker's steps are at least ~0.4 s apart, and an arm
swing takes ~1 s. **Choosing those numbers IS the feature engineering.**


In [ ]:
if describe(ecg_clean) is None or peaks is None:
    print("finish Missions 1 and 2 first")
else:
    signals = {
        # name: (signal, sampling rate, find_peaks keyword-arguments)
        "ecg_clean": (ecg_clean, FS, dict(prominence=1.5, distance=int(0.25 * FS))),
        # TODO (4.1): add the three accel axes with sensible rules, e.g. for az_g (steps):
        #   height above the gravity plateau (try 1.1) and distance >= 0.4 s
        # and for ax_g / ay_g (arm swings): a prominence around 0.5 and distance >= 0.7 s
    }

    rows = []
    for name, (sig, fs, kw) in signals.items():
        p, _ = find_peaks(sig, **kw)
        s = describe(sig)
        dur = len(sig) / fs
        rows.append({"signal": name, **{k: round(v, 3) for k, v in s.items()},
                     "n_peaks": len(p), "rate_per_min": round(len(p) / dur * 60, 1)})
    features = pd.DataFrame(rows).set_index("signal")
    print(features.to_string())
    if len(features) < 4:
        print("\n(4.1 not finished — the table should have four rows)")
    else:
        print("\nRead the rate column: ~110 events/min is a heart, ~108 is a walker's cadence,")
        print("~54 is an arm. One column, three different physical processes.")

---
## Stretch goals (optional, recommended)

**S1 — the hidden delay.** Delay a copy of the ECG by exactly 120 samples, then recover that number
with cross-correlation *(slide 13)*:


In [ ]:
# TODO (S1): a = ecg_clean[:-120]; b = ecg_clean[120:]
#   xc  = np.correlate(b, a, mode="full")
#   lag = xc.argmax() - (len(a) - 1)     -> should be ±120 (sign = which signal leads)


**S2 — parameter courage.** Re-run 2b with prominence 0.5, 1.0, 2.0, 3.0 and plot beats-found
vs prominence. Where is the plateau, and why is a plateau exactly what a trustworthy parameter looks like?


In [ ]:
# TODO (S2): loop over prominences, collect len(peaks), plot the curve


---
## Deliverable 3 — one insight

*2–3 sentences: something the NUMBERS told you today that staring at the waveform plots alone would
not have — cite the actual values.*

*(type your insight here)*

---
## Before you submit (checklist)

- [ ] Whole notebook runs top to bottom — **Runtime → Restart and run all must pass**
- [ ] Mission 1: `describe()` works and the RMS² = mean² + σ² checkpoint prints `True`
- [ ] Mission 2: 2b implemented; the reference checkpoint and honesty plot both shown (Deliverable 2)
- [ ] Mission 3: three r values + the three scatter plots
- [ ] Mission 4: the four-row feature table prints (Deliverable 1)
- [ ] Q1, Q2 answered + the insight written (Deliverable 3)
- [ ] Pushed to your GitHub repository

*Next week: **Looking into Frequency** — the Fourier Transform. Preview with your own numbers: the
1.8 steps/s cadence you measured today will reappear next week as a sharp spectral line at exactly
1.8 Hz. Same signal, new face. And remember: Assignment 1 is live in the LMS — your feature table
today is a rehearsal for it.*
